# Rocket Flight Capstone

The four lessons in this module steered through a complete journey from a physical model to a numerical result that can support an engineering judgment. Writing code was one necessary but not sufficient component. You reach a trustworthy computation over a thoughtful build-up of technical steps.

So far you have: 
- derived ordinary differential equations,
- reconstructed transparent time-stepping methods,
- located events that fall between stored times,
- studied convergence, and
- compared methods at a required accuracy.

You also practiced specifying bounded work for an agent and auditing the code it proposed. This is a key new technical skill in the era of agentic AI, and like any other, it requires practice. That is the purpose of this exercise.

This capstone brings all these ideas together in a new setting: the vertical flight of a small rocket. The physical model involves gravity, aerodynamic drag, changing propellant mass, and known transitions from powered flight to coasting flight, reaching apogee, and falling back to hit the ground. 

You will first work with a fixed-step RK2 calculation and then compare it with an adaptive solver from the [SciPy](https://scipy.org) library. Neither result is authoritative simply because the code runs or comes from a standard library. Your task as always is to decide what evidence makes the reported flight quantities trustworthy.

(rocket-engineering-brief)=
## Engineering brief

A simulation team needs reference results for an idealized vertical-flight model. These results will be used to check later implementations of the same mathematical model. The quantities of interest are the rocket's maximum speed, apogee, and impact conditions.

The engineering question is:

> Can a fixed-step RK2 calculation and a properly configured adaptive SciPy solver support trustworthy predictions of the rocket's apogee and impact conditions to the required numerical accuracy?

Use the following acceptance targets:

- apogee altitude to within $1\ \mathrm{m}$;
- apogee time to within $0.02\ \mathrm{s}$;
- impact time to within $0.05\ \mathrm{s}$; and
- impact speed to within $0.1\ \mathrm{m/s}$.

These are targets for **numerical error within the stated model**. They do not claim that this simplified model predicts a real rocket to the same accuracy.

Your final verdict will need to identify the evidence supporting an RK2 step size and a SciPy tolerance configuration, explain how burnout and the flight events were handled, and separate numerical uncertainty from limitations of the model. Code generation may be assisted, but you remain responsible for recording the specification of delegated work, auditing the result, and deciding what the evidence supports.

## The vertical-flight problem

We model the rocket as a point mass constrained to move vertically. Altitude $h$ is measured upward from the launch point, and velocity $v=dh/dt$ is positive during ascent and negative during descent. The rocket launches from rest at ground level with $100\ \mathrm{kg}$ of propellant. Here are all the problem settings:

| Symbol | Description | Value |
| :--- | :--- | ---: |
| $m_s$ | dry mass of the rocket shell| $50\ \mathrm{kg}$ |
| $m_{p,0}$ | initial propellant mass | $100\ \mathrm{kg}$ |
| $g$ | gravitational acceleration | $9.81\ \mathrm{m/s^2}$ |
| $\rho$ | air density | $1.091\ \mathrm{kg/m^3}$ |
| $r$ | rocket radius | $0.5\ \mathrm{m}$ |
| $A=\pi r^2$ | reference area | $\pi(0.5\ \mathrm{m})^2$ |
| $C_D$ | drag coefficient | $0.15$ |
| $v_e$ | effective exhaust speed relative to the rocket | $325\ \mathrm{m/s}$ |
| $\mu_0$ | powered-flight propellant burn rate | $20\ \mathrm{kg/s}$ |

The idealized model assumes constant gravity, air density, drag coefficient, reference area, exhaust speed, and burn rate during powered flight. It neglects wind, lateral motion, atmospheric variation, rocket attitude, and any change in aerodynamic properties. The engine switches off when the propellant is exhausted.

## Propellant burn rate

A **positive** burn rate $\mu(t)$ removes propellant mass from the rocket, thus $dm_p/dt=-\mu(t)$. With the stated initial mass and constant powered-flight burn rate, burnout occurs at

$$
\label{eq-rocket-burnout-time}
t_b=\frac{m_{p,0}}{\mu_0}=5\ \mathrm{s}.
$$

The prescribed mass-flow history is a step function:

$$
\label{eq-rocket-mass-flow}
\mu(t)=
\begin{cases}
\mu_0, & 0\leq t<t_b,\\
0, & t\geq t_b.
\end{cases}
$$

The value changes discontinuously at the known burnout time, $t_b$. You should treat burnout as an explicit boundary between two integrations, rather than allowing one numerical step to cross it unnoticed.

Let's begin by loading the numerical Python libraries, and setting up the problem. Then make a plot of the burn rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model parameters.
m_s = 50.0        # dry mass (kg)
m_p0 = 100.0      # initial propellant mass (kg)
g = 9.81          # gravitational acceleration (m/s**2)
rho = 1.091       # air density (kg/m**3)
r = 0.5           # rocket radius (m)
A = np.pi * r**2  # reference area (m**2)
C_D = 0.15        # drag coefficient
v_e = 325.0       # effective exhaust speed (m/s)
mu_0 = 20.0       # powered-flight burn rate (kg/s)
t_burn = m_p0 / mu_0

def mass_flow_rate(t):
    '''Return the positive propellant burn rate at time t.'''
    t = np.asarray(t)
    return np.where((t >= 0.0) & (t < t_burn), mu_0, 0.0)

In [ ]:
t_plot = np.linspace(0.0, 10.0, 201)

fig, ax = plt.subplots(figsize=(5.0, 3.0))
ax.step(t_plot, mass_flow_rate(t_plot), where='post')
ax.axvline(t_burn, color='tab:blue', linestyle='--', label='burnout')
ax.set_xlabel('Time, $t$ (s)')
ax.set_ylabel(r'Burn rate, $\mu$ (kg/s)')
ax.set_xlim(0.0, 10.0)
ax.set_ylim(-1.0, 25.0)
ax.grid()
ax.legend()
fig.tight_layout()

Integrating [Equation %s](#eq-rocket-mass-flow) gives the exact remaining propellant mass:

$$
\label{eq-rocket-propellant-history}
m_p(t)=
\begin{cases}
m_{p,0}-\mu_0t, & 0\leq t<t_b,\\
0, & t\geq t_b.
\end{cases}
$$

This exact history will provide a simple but important check of the numerical state.

## Derivation of the equations of motion

:::{warning .simple .dropdown icon=false open=false} On paper — reproduce the model derivation
Follow the derivation below with upward as the positive direction. Reproduce the mass balance, draw and label the forces on the rocket, and write the final three-equation initial-value problem in your paper record. Mark the sign of every force and write the units of each term in Newton's second law.

Keep this derivation beside your computational notebook. During the capstone checkout, you may be asked to explain one step or sign choice.
:::

### Mass balance

The rocket's instantaneous mass is the sum of its constant dry mass and remaining propellant mass:

$$
\label{eq-rocket-total-mass}
m(t)=m_s+m_p(t).
$$

During a short interval $dt$, a positive mass $dm_e=\mu(t)dt$ leaves the rocket. Consequently,

$$
\label{eq-rocket-propellant-balance}
dm_p=-dm_e=-\mu(t)dt,
\qquad
\frac{dm_p}{dt}=-\mu(t).
$$

Using a separate symbol $\mu$ for the positive outflow rate avoids a common sign ambiguity: $\mu$ is positive while propellant is burning, whereas $dm_p/dt$ is negative.

### Newton's second law and thrust

The engine expels propellant downward at the positive rate $\mu(t)$ and effective exhaust speed $v_e$ relative to the rocket. In this model, the corresponding upward thrust is

$$
\label{eq-rocket-thrust}
T(t)=\mu(t)v_e.
$$

Once the exhaust momentum flux is represented by the thrust, Newton's second law for the remaining rocket is simply

$$
\label{eq-rocket-newton-second-law}
m\frac{dv}{dt}=T+F_{\mathrm{ext}},
$$

where $F_{\mathrm{ext}}$ is the sum of gravity and aerodynamic drag. The effective exhaust-speed model incorporates engine-flow details that we do not resolve separately.

:::{note .dropdown icon=false open=false} Why is the thrust $T=\mu v_e$?
During a short interval $dt$, a positive mass $dm_e=\mu dt$ leaves the rocket. The rocket begins with mass $m$ and velocity $v$; it ends with mass $m-dm_e$ and velocity $v+dv$. To first order, the expelled mass has velocity $v-v_e$ in the stationary reference frame. A momentum balance gives

$$
\label{eq-rocket-short-time-momentum}
mv+F_{\mathrm{ext}}dt
=(m-dm_e)(v+dv)+dm_e(v-v_e).
$$

After expanding, canceling equal terms, and neglecting the second-order product $dm_e\,dv$,

$$
m\frac{dv}{dt}=F_{\mathrm{ext}}+\mu v_e.
$$

The last term is the upward momentum delivered to the rocket per unit time, which is the thrust.
:::

### Gravity and aerodynamic drag

Gravity acts downward with force $-mg$. Quadratic drag has magnitude $\tfrac12\rho AC_Dv^2$ and always opposes the motion. In one signed velocity component, both directions are represented by

$$
\label{eq-rocket-drag-force}
F_D=-\frac{1}{2}\rho AC_Dv|v|.
$$

When $v>0$, the drag force is negative and opposes ascent. When $v<0$, $v|v|<0$, so the drag force is positive and opposes descent. Replacing $v|v|$ by $v^2$ would silently give the wrong direction during descent.

The total external force is therefore

$$
\label{eq-rocket-external-force}
F_{\mathrm{ext}}=-mg-\frac{1}{2}\rho AC_Dv|v|.
$$

### The initial-value problem

Substituting [Equation %s](#eq-rocket-thrust) and [Equation %s](#eq-rocket-external-force) into [Equation %s](#eq-rocket-newton-second-law), using $m=m_s+m_p$, and adding the altitude and propellant equations gives

$$
\label{eq-rocket-state-equations}
\begin{aligned}
\frac{dh}{dt} &= v,\\
\frac{dv}{dt} &= -g
+\frac{\mu(t)v_e}{m_s+m_p}
-\frac{\rho AC_D}{2(m_s+m_p)}v|v|,\\
\frac{dm_p}{dt} &= -\mu(t).
\end{aligned}
$$

The three initial conditions are

$$
\label{eq-rocket-initial-conditions}
h(0)=0,
\qquad
v(0)=0,
\qquad
m_p(0)=m_{p,0}=100\ \mathrm{kg}.
$$

In vector form, with $u=[h,v,m_p]^T$, the model has the non-autonomous form $u'=f(t,u)$. The explicit appearance of time through $\mu(t)$ means that the RK2 step used later must evaluate the second stage at both the midpoint state and the midpoint time.

## Flight regimes and reported events

The solution passes through three physical regimes and three consequential transitions:

| Regime or transition | Numerical condition | What changes or is recorded |
| :--- | :--- | :--- |
| Powered ascent | $0\leq t<t_b$ | $\mu=\mu_0$; mass decreases and thrust acts |
| Burnout | known breakpoint $t=t_b$ | set $m_p=0$, record the state, and continue with $\mu=0$ |
| Coasting ascent | $t>t_b$ and $v>0$ | mass is constant; gravity and drag slow the rocket |
| Apogee | first crossing from $v>0$ to $v\leq0$ | interpolate the time and altitude where $v=0$ |
| Descent | $v<0$ and $h>0$ | gravity acts downward and drag acts upward |
| Impact | first downward crossing from $h>0$ to $h\leq0$ | interpolate the time and velocity where $h=0$ |

Burnout is a **breakpoint known in advance**, while apogee and impact are **events discovered from the computed solution**. Impact must mean the downward ground crossing after the rocket has been aloft; the initial condition $h(0)=0$ is the launch point, not an immediate impact.

## Predictions and checks before computing

A numerical trajectory should be judged against expectations that do not come from the same computation. Three useful checks follow directly from the model.

**Exact propellant history.** Use [Equation %s](#eq-rocket-propellant-history) to calculate the remaining propellant at $t=3.2\ \mathrm{s}$ and confirm that it reaches exactly zero at burnout.

**No-drag burnout-speed bound.** If drag is removed during powered flight, integrating the velocity equation gives

$$
\label{eq-rocket-ideal-burn-velocity}
v_{\mathrm{ideal}}(t)
=v_e\ln\left(\frac{m_s+m_{p,0}}{m_s+m_p(t)}\right)-gt.
$$

During ascent, drag can only reduce the velocity from this ideal value. Evaluate [Equation %s](#eq-rocket-ideal-burn-velocity) at burnout and use it as an upper bound for the powered-flight speed.

**Empty-shell terminal speed.** During descent after burnout, the mass is $m_s$ and a steady downward velocity satisfies $dv/dt=0$. Its magnitude is

$$
\label{eq-rocket-terminal-speed}
v_{\mathrm{terminal}}
=\sqrt{\frac{2m_sg}{\rho AC_D}}.
$$

The rocket begins its descent from rest at apogee, so its impact-speed magnitude should approach this value from below.

Finally, every term in [Equation %s](#eq-rocket-newton-second-law) has units of force: $mg$, $\mu v_e$, and $\rho Av^2$ each have units $\mathrm{kg\,m/s^2}$. This dimensional agreement is necessary, although it cannot by itself establish that every sign or coefficient is correct.

:::{warning .simple .dropdown icon=false open=false} On paper — record independent expectations
Before beginning the numerical calculation, your paper record should contain:

- the mass balance, thrust relation, and Newton's-law model;
- the initial-value problem with the state order and sign convention;
- a hand-drawn diagram of the burn, coast, descent, and impact sequence;
- the remaining propellant at $3.2\ \mathrm{s}$;
- the no-drag burnout-speed bound and empty-shell terminal speed; and
- your predicted sign for every force term during ascent and descent.

These expectations become evidence when you later audit the RK2 and SciPy calculations.
:::

## Build a time-aware RK2 flight solver

The state equation $u'=f(t,u)$ depends explicitly on time because the mass-flow rate changes at burnout. The numerical calculation therefore needs three distinct layers:

| Layer | Responsibility | Question it answers |
| :--- | :--- | :--- |
| Model right-hand side | translate the three differential equations into derivatives | What is the instantaneous rate of change? |
| RK2 step | advance the complete state from one time to the next | How is one numerical update formed? |
| Flight driver | repeat steps, stop at the known breakpoint, and locate events | How is a complete flight assembled and reported? |

Keeping these responsibilities separate makes each part easier to inspect and test.

:::{warning .simple .dropdown icon=false open=false} In your notebook — reconstruct the RK2 flight solver
Create a section with the same title in your working notebook. Reconstruct the four function-definition blocks below one at a time, in order. Before entering each block, write a one-sentence specification of its purpose, inputs, and output. Run the cell that defines the function, then compare your version line by line with the published version before continuing. After the definitions, reconstruct and run the diagnostic cells as instructed.

If you use an agent to help reproduce a block, save the instruction you gave it. Bound the request to that one named function and its stated interface, and require the agent to leave all other cells unchanged. You still need to audit every line against the equations and explain the result.

Do not use **Run All** while reconstructing. A definition that happens to execute is not yet evidence that its logic is correct.
:::

### Translate the model into a right-hand side

The right-hand-side function returns the derivatives in the same order as the state $u=[h,v,m_p]^T$. Naming the three forces separately makes their signs inspectable before they are combined into the acceleration. The mass-flow function is passed as an argument so that the time dependence is an explicit dependency of the model.

In [ ]:
def rocket_rhs(t, u, mass_flow, m_s, g, rho, A, C_D, v_e):
    '''Return [dh/dt, dv/dt, dm_p/dt] for the rocket model.

    Parameters
    ----------
    t : float
        Current time in seconds.
    u : numpy.ndarray
        Current state [altitude, velocity, propellant mass].
    mass_flow : callable
        Function returning the positive propellant burn rate at time t.
    m_s, g, rho, A, C_D, v_e : float
        Physical parameters of the rocket model.

    Returns
    -------
    numpy.ndarray
        Derivatives in the same order as u.
    '''
    h, v, m_p = u
    total_mass = m_s + m_p
    mu = float(mass_flow(t))

    thrust_force = mu * v_e
    weight_force = -total_mass * g
    drag_force = -0.5 * rho * A * C_D * v * abs(v)

    altitude_rate = v
    acceleration = (thrust_force + weight_force + drag_force) / total_mass
    propellant_rate = -mu

    return np.array([altitude_rate, acceleration, propellant_rate])

:::{note .simple .dropdown icon=false open=false} Self-check — explain the model translation
You should be ready to answer these questions, if asked:
- What state order does `rocket_rhs()` require, and where is that order preserved?
- Why does `drag_force` have the correct sign during both ascent and descent?
- Why is $\mu$ positive while `propellant_rate` is negative?
:::

### Advance one time-aware midpoint step

Lesson 4 introduced explicit-midpoint RK2 for an autonomous system (meaning, time does not appear explicitly on the right-hand side). For $u'=f(t,u)$, both the midpoint state and midpoint time must be used. Rewrite the RK2 algorithm in this textbook form:

$$
\label{eq-rocket-rk2-time-aware}
\begin{aligned}
k_1 &= f(t_n,u_n),\\
u_{n+1/2} &= u_n+\frac{\Delta t}{2}k_1,\\
k_2 &= f\left(t_n+\frac{\Delta t}{2},u_{n+1/2}\right),\\
u_{n+1} &= u_n+\Delta t\,k_2.
\end{aligned}
$$

The function below is still a single numerical step. It knows nothing about burnout, apogee, or impact.  In particular, shortening a step to land exactly at burnout belongs in the driver; it is not part of the RK2 formula.

In [ ]:
def rk2_step(t, u, f, dt, *args):
    '''Return one explicit-midpoint RK2 step for u' = f(t, u).'''
    slope_start = f(t, u, *args)

    t_midpoint = t + 0.5 * dt
    u_midpoint = u + 0.5 * dt * slope_start
    slope_midpoint = f(t_midpoint, u_midpoint, *args)

    return u + dt * slope_midpoint

:::{note .simple .dropdown icon=false open=false} Self-check — explain one RK2 step
You should be ready to answer these questions, if asked:
- At what time and state is each call to `f()` evaluated?
- Why does the final update start from `u`, rather than from `u_midpoint`?
- Which line would be wrong if the model depended on time but the autonomous Lesson 4 step were copied unchanged?
:::

### Interpolate a bracketed event

A step brackets an event when the monitored component is positive at the beginning and nonpositive at the end. The same linear interpolation used for touchdown in Lesson 4 estimates the event time and the complete state. This helper function assumes that the driver has already checked the bracket.

In [ ]:
def interpolate_zero_crossing(t, u, t_next, u_next, component):
    '''Linearly interpolate the state where one component reaches zero.'''
    value = u[component]
    value_next = u_next[component]
    fraction = value / (value - value_next)

    event_time = t + fraction * (t_next - t)
    event_state = u + fraction * (u_next - u)
    return event_time, event_state

### Drive the calculation through burnout and flight events

The driver normally advances by the requested fixed step $\Delta t$. It shortens a step only when necessary to end exactly at the known burnout time or at the time limit. This is **breakpoint handling**, not adaptive error control.

After each completed step, the driver looks for the first downward crossing of $v=0$ and then the first downward crossing of $h=0$. It stores interpolated apogee and impact states and returns a status rather than assuming that impact must have occurred.

In [ ]:
def integrate_rocket_rk2(u_0, dt, time_limit, t_burn, *rhs_args):
    '''Integrate the rocket through burnout, apogee, and impact.'''
    if dt <= 0.0:
        raise ValueError('dt must be positive.')
    if time_limit <= t_burn:
        raise ValueError('time_limit must extend beyond burnout.')

    t = 0.0
    u = np.array(u_0, dtype=float)
    times = [t]
    states = [u.copy()]
    steps = 0
    status = 'time_limit'

    burnout_state = None
    apogee_time = None
    apogee_state = None
    impact_time = None
    impact_state = None

    while t < time_limit:
        dt_step = min(dt, time_limit - t)

        # End a powered-flight step exactly at the known breakpoint.
        ending_at_burnout = t < t_burn <= t + dt_step
        if ending_at_burnout:
            dt_step = t_burn - t

        u_next = rk2_step(t, u, rocket_rhs, dt_step, *rhs_args)
        t_next = t + dt_step
        steps += 1

        if ending_at_burnout:
            t_next = t_burn
            # The exact mass balance gives zero propellant at burnout.
            u_next[2] = 0.0
            burnout_state = u_next.copy()

        if not np.all(np.isfinite(u_next)) or u_next[2] < -1e-12:
            status = 'invalid_state'
            break

        # Apogee is the first crossing from upward to downward velocity.
        if apogee_time is None and u[1] > 0.0 and u_next[1] <= 0.0:
            apogee_time, apogee_state = interpolate_zero_crossing(
                t, u, t_next, u_next, component=1
            )

        # Requiring h > 0 at the start avoids treating launch as impact.
        if u[0] > 0.0 and u_next[0] <= 0.0:
            impact_time, impact_state = interpolate_zero_crossing(
                t, u, t_next, u_next, component=0
            )
            times.append(impact_time)
            states.append(impact_state.copy())
            status = 'impact'
            break

        times.append(t_next)
        states.append(u_next.copy())
        t, u = t_next, u_next

    times = np.array(times)
    states = np.array(states)
    speed_index = np.argmax(np.abs(states[:, 1]))

    return {
        'status': status,
        'times': times,
        'states': states,
        'steps': steps,
        'rhs_evaluations': 2 * steps,
        'burnout_time': t_burn,
        'burnout_state': burnout_state,
        'apogee_time': apogee_time,
        'apogee_state': apogee_state,
        'impact_time': impact_time,
        'impact_state': impact_state,
        'maximum_speed': abs(states[speed_index, 1]),
        'maximum_speed_time': times[speed_index],
    }

:::{note .simple .dropdown icon=false open=false} Self-check — explain the flight logic
You should be ready to answer these questions, if asked:
- Why is burnout handled by ending a step at $t_b$, rather than by stepping past it and clamping a negative mass?
- What is the difference between the known burnout breakpoint and the discovered apogee and impact events?
- Which condition prevents the initial state $h(0)=0$ from being reported as impact?
- What assumption is made when an event is interpolated between two RK2 states?
:::

### Make one provisional flight

Use $\Delta t=0.6\ \mathrm{s}$ for a first diagnostic run. This value deliberately does not divide the $5\ \mathrm{s}$ burn time, so the result exposes whether the driver lands on the breakpoint as intended. It is a provisional step, not a claim that the engineering accuracy targets have been met. A $60\ \mathrm{s}$ time limit comfortably allows the anticipated trajectory to reach the ground.

Before running the cell, predict the result status and the order of burnout, maximum speed, apogee, and impact.

In [ ]:
u_0 = np.array([0.0, 0.0, m_p0])
dt_trial = 0.6      # deliberately does not divide the burnout time
time_limit = 60.0  # seconds
rhs_args = (mass_flow_rate, m_s, g, rho, A, C_D, v_e)

rk2_result = integrate_rocket_rk2(
    u_0, dt_trial, time_limit, t_burn, *rhs_args
)

times = rk2_result['times']
states = rk2_result['states']
propellant_at_3_2 = np.interp(3.2, times, states[:, 2])

print(f'Status: {rk2_result["status"]}')
print(f'Completed steps: {rk2_result["steps"]}')
print(f'RHS evaluations: {rk2_result["rhs_evaluations"]}')
print(f'Propellant at 3.2 s: {propellant_at_3_2:.6f} kg')

if rk2_result['burnout_state'] is not None:
    h_burn, v_burn, m_p_burn = rk2_result['burnout_state']
    print(
        f'Burnout: t = {rk2_result["burnout_time"]:.6f} s, '
        f'h = {h_burn:.6f} m, v = {v_burn:.6f} m/s, '
        f'm_p = {m_p_burn:.6f} kg'
    )

print(
    f'Maximum speed: {rk2_result["maximum_speed"]:.6f} m/s '
    f'at t = {rk2_result["maximum_speed_time"]:.6f} s'
)

if rk2_result['apogee_state'] is not None:
    print(
        f'Apogee: t = {rk2_result["apogee_time"]:.6f} s, '
        f'h = {rk2_result["apogee_state"][0]:.6f} m'
    )

if rk2_result['impact_state'] is not None:
    print(
        f'Impact: t = {rk2_result["impact_time"]:.6f} s, '
        f'v = {rk2_result["impact_state"][1]:.6f} m/s'
    )

Plot the three state components against time. Use the figure as a diagnostic: look for the expected linear decrease of propellant during the burn, continuity of altitude and velocity at burnout, one apogee, and a return to $h=0$. A smooth-looking curve cannot establish numerical accuracy.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(5.0, 5.0), sharex=True)

axes[0].plot(times, states[:, 0])
axes[0].set_ylabel('Altitude, $h$ (m)')

axes[1].plot(times, states[:, 1])
axes[1].axhline(0.0, color='0.7', linewidth=1.0)
axes[1].set_ylabel('Velocity, $v$ (m/s)')

axes[2].plot(times, states[:, 2])
axes[2].set_ylabel('Propellant, $m_p$ (kg)')
axes[2].set_xlabel('Time, $t$ (s)')

for ax in axes:
    ax.axvline(t_burn, color='tab:orange', linestyle='--')
    ax.grid()

fig.tight_layout()

### Decide what the first run supports

Develop preliminary confidence by combining evidence that would fail in different ways:

1. **Code-to-equation inspection:** point from each returned derivative to the matching term in [Equation %s](#eq-rocket-state-equations), including its sign.
2. **Exact mass check:** compare the reported propellant at $3.2\ \mathrm{s}$ and at burnout with your paper values. Confirm that burnout occurs at exactly $5\ \mathrm{s}$ even though $0.6\ \mathrm{s}$ does not divide it.
3. **Event logic:** confirm that the status is `impact`, the event order is physically sensible, the apogee state has $v=0$, and the impact state has $h=0$.
4. **Independent bounds:** compare the maximum speed with the no-drag burnout-speed bound and the impact-speed magnitude with the empty-shell terminal speed from your paper record.
5. **Trajectory inspection:** use the plot to look for wrong signs, jumps, repeated events, or propellant used after burnout.

Record any discrepancy before changing code. Passing these checks means that the calculation has survived useful attempts to expose an obvious defect. It does **not** show that $\Delta t=0.6\ \mathrm{s}$ meets the acceptance targets, and agreement of printed digits from one run is not an accuracy argument. Keep this result unchanged as the baseline for the verification and refinement work that follows later.

:::{note .simple .dropdown icon=false open=false} Self-check — defend your confidence
You should be ready to answer these questions, if asked:
- Choose two checks above and explain what different mistake each could reveal.
- Which claims can you make from this one run, and which claims still require refinement or an independent solver?
:::

---

The narrative and instructional content of this notebook are licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Code cells are licensed under the [BSD 3-Clause License](../../../LICENSES/BSD-3-Clause.txt).